In [ ]:
# Hugging Face: https://huggingface.co/datasets/Koolenbrander/assignment-2

from datasets import load_dataset

ds = load_dataset("aegean-ai/rav4-exterior-images", split="train")

/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 65/65 [00:00<00:00, 257.32 examples/s]


In [ ]:
import bentoml
from ultralytics import YOLO
# https://docs.ultralytics.com/datasets/segment/carparts-seg/
model = YOLO('yolov8n.pt') 
model.train(data='carparts-seg.yaml', epochs=5, imgsz=640, workers=0)


Ultralytics 8.4.14 🚀 Python-3.11.13 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=carparts-seg.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x729b43e60150>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([0.        , 0.001001  , 0.002002  , 0.003003  , 0.004004  ,
       0.00500501, 0.00600601, 0.00700701, 0.00800801, 0.00900901,
       0.01001001, 0.01101101, 0.01201201, 0.01301301, 0.01401401,
       0.01501502, 0.01601602, 0.01701702, 0.01801802, 0.01901902,
       0.02002002, 0.02102102, 0.02202202, 0.02302302, 0.02402402,
       0.02502503, 0.02602603, 0.02702703, 0.02802803, 0.02902903,
       0.03003003, 0.03103103, 0.03203203, 0.03303303, 0.03403403,
       0.03503504, 0.03603604, 0.03703704, 0.03803804, 0.03903904,
       0.04004004, 0.04104

In [6]:
model = YOLO('runs/detect/train2/weights/best.pt')
results = model('frames/')


image 1/47 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0001.jpg: 384x640 1 back_glass, 1 front_glass, 87.9ms
image 2/47 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0002.jpg: 384x640 1 back_glass, 1 back_light, 1 hood, 28.1ms
image 3/47 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0003.jpg: 384x640 1 hood, 27.3ms
image 4/47 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0004.jpg: 384x640 1 back_bumper, 1 back_light, 3 hoods, 27.3ms
image 5/47 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0005.jpg: 384x640 1 back_glass, 2 hoods, 26.2ms
image 6/47 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0006.jpg: 384x640 1 front_glass, 1 hood, 26.2ms
image 7/47 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0007.jpg: 384x640 1 front_right_light, 1 left_mirror, 1 right_mirror, 26.5ms
image 8/47 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0008.jpg: 384x6

In [ ]:
import os
import pandas as pd

detections = []
frame_folder = 'frames/'
for filename in sorted(os.listdir(frame_folder)):
    if filename.endswith(".jpg"):
        frame_number = int(filename.split('_')[1].split('.')[0])
        timestamp_in_seconds = frame_number 
        results = model(os.path.join(frame_folder, filename))
        
        for r in results:
            for box in r.boxes:
                class_id = int(box.cls)
                conf = float(box.conf)
                coords = box.xyxy[0].tolist() 

                detections.append({
                    "video_id": "YcvECxtXoxQ",
                    "timestamp": timestamp_in_seconds,
                    "class_label": model.names[class_id],
                    "bounding_box": coords,
                    "confidence_score": conf
                })

df = pd.DataFrame(detections)


image 1/1 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0001.jpg: 384x640 1 back_glass, 1 front_glass, 29.4ms
Speed: 6.0ms preprocess, 29.4ms inference, 3.9ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0002.jpg: 384x640 1 back_glass, 1 back_light, 1 hood, 29.3ms
Speed: 4.8ms preprocess, 29.3ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0003.jpg: 384x640 1 hood, 29.5ms
Speed: 4.5ms preprocess, 29.5ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0004.jpg: 384x640 1 back_bumper, 1 back_light, 3 hoods, 27.3ms
Speed: 5.3ms preprocess, 27.3ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /workspaces/eng-ai-agents/assignments/assignment-2/frames/frame_0005.jpg: 384x640 1 back_g

In [8]:
df.to_parquet('video_detections.parquet')

In [ ]:
from datasets import load_dataset

ds = load_dataset("aegean-ai/rav4-exterior-images", split="train")

def get_query_parts(image):
    results = model(image)
    parts_found = []
    for r in results:
        for box in r.boxes:
            label = model.names[int(box.cls)]
            parts_found.append(label)
    return list(set(parts_found))

/workspaces/eng-ai-agents/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def retrieve_clips(query_label, df):

    matches = df[df['class_label'] == query_label].copy()
    
    timestamps = sorted(matches['timestamp'].unique())
    
    if not timestamps:
        return []

    clips = []
    if timestamps:
        start = timestamps[0]
        for i in range(1, len(timestamps)):
            if timestamps[i] - timestamps[i-1] > 2:
                clips.append((start, timestamps[i-1]))
                start = timestamps[i]
        clips.append((start, timestamps[-1]))
        
    return clips

In [ ]:

query_img = ds[0]['image']
detected_parts = get_query_parts(query_img)

for part in detected_parts:
    intervals = retrieve_clips(part, df)
    for start, end in intervals:
        print(f"--- Result for {part} ---")
        print(f"Start: {start}s | End: {end}s")
        print(f"Class: {part}")
        print(f"Supporting Detections: {len(df[(df['timestamp'] >= start) & (df['timestamp'] <= end)])}")

0: 384x640 1 back_right_door, 1 front_door, 1 front_glass, 1 front_right_door, 1 hood, 1 left_mirror, 2 right_mirrors, 2 wheels, 29.1ms
Speed: 4.6ms preprocess, 29.1ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
--- Result for wheel ---
Start: 25s | End: 25s
Class: wheel
Supporting Detections: 3
--- Result for wheel ---
Start: 39s | End: 39s
Class: wheel
Supporting Detections: 5
--- Result for hood ---
Start: 2s | End: 6s
Class: hood
Supporting Detections: 14
--- Result for hood ---
Start: 11s | End: 12s
Class: hood
Supporting Detections: 11
--- Result for hood ---
Start: 22s | End: 32s
Class: hood
Supporting Detections: 27
--- Result for hood ---
Start: 36s | End: 47s
Class: hood
Supporting Detections: 48
--- Result for left_mirror ---
Start: 7s | End: 7s
Class: left_mirror
Supporting Detections: 3
--- Result for right_mirror ---
Start: 7s | End: 8s
Class: right_mirror
Supporting Detections: 7
--- Result for right_mirror ---
Start: 34s | End: 35s
Class: right_mirr

In [ ]:
from huggingface_hub import HfApi, login


login(token="lol")

api = HfApi()
repo_id = "Koolenbrander/assignment-2"

api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

RepoUrl('https://huggingface.co/datasets/Koolenbrander/assignment-2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Koolenbrander/assignment-2')

In [15]:
api.upload_file(
    path_or_fileobj="video_detections.parquet",
    path_in_repo="video_detections.parquet",
    repo_id=repo_id,
    repo_type="dataset"
)

Processing Files (1 / 1): 100%|██████████| 9.61kB / 9.61kB, 47.9kB/s  
New Data Upload: 100%|██████████| 9.61kB / 9.61kB, 47.9kB/s  


CommitInfo(commit_url='https://huggingface.co/datasets/Koolenbrander/assignment-2/commit/bcc29da423e89352969b892a99d09e65f8ee223b', commit_message='Upload video_detections.parquet with huggingface_hub', commit_description='', oid='bcc29da423e89352969b892a99d09e65f8ee223b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Koolenbrander/assignment-2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Koolenbrander/assignment-2'), pr_revision=None, pr_num=None)

In [17]:
api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="dataset"
)

CommitInfo(commit_url='https://huggingface.co/datasets/Koolenbrander/assignment-2/commit/251faf8a996f40ed694fbe12012a94d9fb9cd7ab', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='251faf8a996f40ed694fbe12012a94d9fb9cd7ab', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Koolenbrander/assignment-2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Koolenbrander/assignment-2'), pr_revision=None, pr_num=None)